# 🧠 Neuro-Glymphatic Tau Dynamics (NGTD) — Interactive Bedside Simulator
**A Multiscale Biophysical Reaction-Diffusion & Perivascular Clearance Engine**

**Author**: Luca Celebrano, MD Candidate (Università degli Studi di Napoli Federico II / TIGEM)
**Permanent Archive**: [CERN Zenodo 10.5281/zenodo.24261518](https://doi.org/10.5281/zenodo.24261518)  
**Preprint Status**: Under Review / In Submission (Research Square)
**GitHub Repository**: [Lukecele/tau-glymphatic-continuum](https://github.com/Lukecele/tau-glymphatic-continuum)  
**License**: CC-BY-4.0 International

---
### Purpose:
This interactive notebook simulates perivascular glymphatic clearance, annular Péclet numbers, and neocortical Tau spreading risk based on patient-specific in vivo **DTI-ALPS** indices and **APOE-ε4** genetic burden.

In [ ]:
# Step 1: Install NGTD engine directly from GitHub
!pip install -q git+https://github.com/Lukecele/tau-glymphatic-continuum.git
print('✓ NGTD Engine successfully installed!')

In [ ]:
# Step 2: Import simulation modules
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

from tau_glymphatic import (
    calculate_effective_velocity,
    calculate_peclet_number,
    solve_alps_critical_threshold,
    calculate_microglial_strain,
    calculate_net_clearance_rate,
    simulate_radial_pvs_profile,
)
from tau_glymphatic.pde_brinkman import (
    solve_annular_stokes_velocity_profile,
    calculate_front_velocity_fisher,
    evaluate_wave_pinning_criterion,
    calculate_dimensionless_hydrodynamic_numbers,
)

print('✓ All simulation modules ready.')

In [ ]:
# Step 3: Interactive Simulation Function
def run_interactive_simulation(alps_index=1.60, apoe_e4=0, trem2_r47h=False, nlrp3=False, sleep_pct=33):
    sleep_frac = sleep_pct / 100.0
    v_eff = calculate_effective_velocity(alps_index, sleep_fraction=sleep_frac)
    pe_sleep = calculate_peclet_number(alps_index, peak_sleep=True)
    crit = solve_alps_critical_threshold()
    alps_crit = crit['alps_crit_kinetic']
    
    rates = calculate_net_clearance_rate(
        alps_index=alps_index,
        apoe_e4_alleles=apoe_e4,
        trem2_r47h=trem2_r47h,
        nlrp3_activated=nlrp3,
    )
    
    k_net = rates['effective_growth_rate']
    v_front = calculate_front_velocity_fisher(3.5156e-12, k_net)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
    
    # Panel 1: Radial Tau Profile
    sim = simulate_radial_pvs_profile(alps_index=alps_index, apoe_e4_alleles=apoe_e4, num_grid_points=50)
    ax1.plot(sim['r_points_um'], sim['c_profile'], color='crimson' if rates['is_pathologic_expansion'] else 'teal', lw=2.5)
    ax1.axvline(10, color='gray', ls='--', alpha=0.6, label='Arteriolar Wall (r0)')
    ax1.axvline(30, color='blue', ls=':', alpha=0.6, label='Glia Limitans (r1)')
    ax1.set_xlabel('Radial Distance r (um)', fontsize=11)
    ax1.set_ylabel('Normalized Seed Concentration C/Cmax', fontsize=11)
    ax1.set_title('Perivascular & Neuropil Radial Profile', fontsize=12, fontweight='bold')
    ax1.set_ylim(0, 1.05)
    ax1.legend(loc='upper left', frameon=True)
    ax1.grid(True, alpha=0.25)
    
    # Panel 2: Bifurcation Space
    alps_range = np.linspace(1.0, 1.8, 100)
    for alleles, col, lab in zip([0, 1, 2], ['#2ca02c', '#ff7f0e', '#d62728'], ['APOE e4 (0)', 'APOE e4 (1)', 'APOE e4 (2)']):
        growth_curve = [calculate_net_clearance_rate(a, apoe_e4_alleles=alleles)['effective_growth_rate'] * 1e6 for a in alps_range]
        ax2.plot(alps_range, growth_curve, color=col, lw=2, label=lab)
    
    ax2.axhline(0, color='black', lw=1.2, ls='-')
    ax2.axvline(alps_crit, color='purple', ls='--', lw=2, label=f'Analytical Tipping ({alps_crit:.2f})')
    ax2.scatter([alps_index], [k_net * 1e6], color='red' if rates['is_pathologic_expansion'] else 'green', s=120, zorder=5, label='Patient State')
    ax2.set_xlabel('DTI-ALPS Index', fontsize=11)
    ax2.set_ylabel('Net Tau Growth Rate dk (10^-6 s^-1)', fontsize=11)
    ax2.set_title('Phase Space Bifurcation & Tipping Boundary', fontsize=12, fontweight='bold')
    ax2.legend(loc='upper right', frameon=True)
    ax2.grid(True, alpha=0.25)
    
    plt.tight_layout()
    plt.show()
    
    # Text Diagnostic Output
    state_icon = '🚨 UNSTABLE (Seeding Wave Advances)' if rates['is_pathologic_expansion'] else '🛡️ STABLE (Wave-Pinned Extinction)'
    pe_desc = 'Convection Dominates' if pe_sleep > 1.0 else 'Diffusion & Stasis Dominates'
    print('='*75)
    print(f'  CLINICAL REGIME: {state_icon}')
    print(f'  • DTI-ALPS Index:               {alps_index:.2f} (Critical Boundary = {alps_crit:.2f})')
    print(f'  • Interstitial Washout Velocity: {v_eff*1e9:.2f} nm/s')
    print(f'  • Fisher Seeding Front Speed:    {v_front*1e9:.2f} nm/s')
    print(f'  • Sleep Péclet Number:           {pe_sleep:.3f} ({pe_desc})')
    print(f'  • Microglial Strain Index:       {rates["microglial_strain"]:.3f} / 1.000')
    print('='*75)

# Launch interactive UI widget
widgets.interact(
    run_interactive_simulation,
    alps_index=widgets.FloatSlider(value=1.60, min=1.00, max=1.80, step=0.01, description='DTI-ALPS:'),
    apoe_e4=widgets.Dropdown(options=[0, 1, 2], value=0, description='APOE-e4:'),
    trem2_r47h=widgets.Checkbox(value=False, description='TREM2 R47H'),
    nlrp3=widgets.Checkbox(value=False, description='NLRP3 High'),
    sleep_pct=widgets.IntSlider(value=33, min=0, max=50, step=1, description='Sleep %:'),
);